In [ ]:
"""
Build Pyrome-specific FlamMap lookups

author: maxwell.cook@colostate.edu
"""

import os, sys

from pathlib import Path
from os.path import join
from fb_tools.weather import (
    load_gridmet_csv, build_flammap_scenario_cache,
    load_flammap_scenario_cache
)

# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

In [ ]:
# --- Load the gridMET climatology by Pyrome
gridmet_fp = Path(join(projdir,"data/tabular/raw/weather/gridmet_clim_CO_pyromes.csv"))
# --- Load and inspect
clim = load_gridmet_csv(gridmet_fp) # parses gridmet columns
print(clim.shape)
print("Pyromes :", sorted(clim["pyrome"].unique()))
print("Years   :", sorted(clim["year"].unique()))
print("Columns :", list(clim.columns))

In [ ]:
print("tmmn_f" in clim.columns, "vpd_pa" in clim.columns)

In [ ]:
# --- Build wind scenario by pyrome
from fb_tools.weather import wind_percentiles_from_cell_cache

# --- Build the wind dataframe from HRRR cache
wind_pcts = wind_percentiles_from_cell_cache(
    cache_dir=join(projdir,'data/weather/pyrome_wind'),
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.97],  # default
)
wind_pcts['42']

In [ ]:
# CO pyromes span ~37–41°N; 39.5° used for GSI photoperiod calculation.
# GSI uses tmmn_f (min temp), vpd_pa (from GEE export), and daylength.
# wind_direction=-2 = downhill (worst-case); override with -1 (uphill)
# or an explicit azimuth (0–360) as needed.

CACHE_DIR  = Path(join(projdir,"data/weather/flammap/"))

percentiles = [0.25, 0.50, 0.75, 0.90, 0.97]

scenarios = build_flammap_scenario_cache(
    clim,
    pyrome_col="pyrome",
    percentiles=percentiles,
    out_dir=CACHE_DIR,
    lat_deg=39.5, # adjust as-needed
    wind_direction=-2, # FlamMap downhill default
    wind_percentiles=wind_pcts,
)
scenarios

In [ ]:
# --- Look up Pyrome
# --- Load the Pyrome-specific FlamMap inputs
from fb_tools.weather import load_flammap_scenario_cache

# --- Pyrome 46 CO Rockies
fm_params = load_flammap_scenario_cache(46, CACHE_DIR)
fm_params

In [ ]:
import matplotlib.pyplot as plt

percentile_labels = [f"p{int(p*100)}" for p in fm_params['percentiles']]
fm_herb  = [fm_params['scenarios'][k]['FM_herb']  for k in percentile_labels]
fm_woody = [fm_params['scenarios'][k]['FM_woody'] for k in percentile_labels]
x = fm_params['percentiles']

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(x, fm_herb,  marker='o', lw=2, color='#2E7D32', label='FM Herb',  zorder=3)
ax.plot(x, fm_woody, marker='s', lw=2, color='#8B4513', label='FM Woody', zorder=3)

# Annotate values
for xi, yh, yw in zip(x, fm_herb, fm_woody):
    ax.annotate(f'{yh:.0f}',  (xi, yh),  textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8, color='#2E7D32')
    ax.annotate(f'{yw:.0f}', (xi, yw), textcoords='offset points', xytext=(0, -14),
                ha='center', fontsize=8, color='#8B4513')

ax.set_xticks(x)
ax.set_xticklabels([f"{int(p*100)}th" for p in x])
ax.set_xlabel('ERC Percentile', fontsize=11)
ax.set_ylabel('Fuel Moisture (%)', fontsize=11)
ax.set_title(f"Live Fuel Moisture by ERC Percentile — Pyrome {fm_params['pyrome_id']}", fontsize=12)
ax.legend(framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_ylim(0, max(fm_woody) * 1.15)

plt.tight_layout()
plt.savefig('fm_by_percentile.png', dpi=150, bbox_inches='tight')
plt.show()

## Region 4 forests — Payette & Wasatch-Uinta-Cache

Build the FlamMap batch table for the two R4 forests from the **RTMA-based**
per-pyrome scenario caches in `data/weather/flammap_rtma/` (written by
`00d_RTMA-FM` §7). **The LCP column is left blank** — this table is handed off
for someone else to fill in each forest's fuelscape path.

**Method**
1. Map each forest to its **dominant pyrome** (greatest boundary overlap):
   - **Payette** → pyrome **14** (Idaho Batholith, 79% of forest; Blue Mtns 16 ≈ 20%)
   - **Wasatch-Uinta-Cache (WUC)** → pyrome **44** (Wasatch-Uinta Mtns, 93%)

   (derivable automatically with `get_pyrome_id(forest_geom, pyromes)` — see the
   commented block in the config cell.)
2. `load_flammap_scenario_cache(pyrome)` → `scenario_cache_to_conditions()` → a
   `conditions` table (one row per ERC percentile p25…p97: RTMA dead FM, dead-FM-
   scaled live FM — monotonic — and HRRR wind speed, `WIND_DIRECTION = -2` downhill).
3. `build_scenarios(conditions, [""])` → the FlamMap scenario rows with a **blank LCP**.
4. Concatenate the forests → one **batch CSV** (the FlamMap "batch file").
5. Downstream: fill the LCP column (one path per Forest), then on the Windows VM
   `run_batch(fm_exe, batch, output_root)` writes each run's `FlamMap.input` +
   `FMcommand.txt` and executes `TestFlamMap.exe`.

In [ ]:
# ── R4 config ─────────────────────────────────────────────────────────────
from fb_tools import scenario_cache_to_conditions, build_scenarios
from fb_tools.weather import load_flammap_scenario_cache

# RTMA-based per-pyrome scenario caches (written by 00d_RTMA-FM §7).
RTMA_CACHE_DIR = projdir / "data/weather/flammap_rtma"

# Forest → dominant pyrome (greatest boundary overlap).
FOREST_PYROME = {
    "Payette": 14,   # Idaho Batholith (79% of forest); Blue Mtns (16) ≈ 20%
    "WUC":     44,   # Wasatch-Uinta Mtns (93% of Uinta-Wasatch-Cache)
}
# The LCP path is left BLANK in the batch (handed off for someone to fill in).

# --- (optional) derive FOREST_PYROME automatically from the forest boundaries:
# import geopandas as gpd
# from fb_tools.utils.geo import get_pyrome_id
# forests = gpd.read_file(projdir / "data/spatial/raw/boundaries/r4_forests_wuc_payette.geojson")
# pyr = gpd.read_file("/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/boundaries/pyromes/Pyromes_CONUS_20200206.shp")
# FOREST_PYROME = {r.forest_id: int(get_pyrome_id(r.geometry, pyr, pyrome_col="PYROME"))
#                  for _, r in forests.iterrows()}
# print("derived:", FOREST_PYROME)

In [ ]:
# ── Build the FlamMap batch: cache → conditions → scenarios, per forest ────
# LCP is left BLANK — this table is handed off for someone else to fill in the
# LCP path per forest. (To make a self-contained runnable table instead, pass
# each forest's LCP path to build_scenarios in place of "".)
import pandas as pd

frames = []
for forest, pid in FOREST_PYROME.items():
    cache = load_flammap_scenario_cache(pid, RTMA_CACHE_DIR)   # RTMA dead FM + dead-scaled live FM + HRRR wind
    cond  = scenario_cache_to_conditions(cache)                # p25..p97 rows
    scen  = build_scenarios(cond, [""])                        # blank LCP
    scen["LCP"]     = ""                                       # <-- fill downstream (one path per Forest)
    scen["FM_NAME"] = forest                                   # label (normally the LCP stem)
    scen.insert(0, "Forest", forest)
    scen.insert(1, "Pyrome", pid)
    frames.append(scen)
    print(f"  {forest:8} pyrome {pid} → {len(scen)} scenarios (LCP blank)")

batch = pd.concat(frames, ignore_index=True)

# Save the batch table — the FlamMap "batch file". Fill the blank LCP column
# (one path per Forest), then run via run_batch on the Windows VM.
batch_fp = projdir / "data/tabular/mod/flammap_batch_R4.csv"
batch_fp.parent.mkdir(parents=True, exist_ok=True)
batch.to_csv(batch_fp, index=False)
print(f"\nWrote FlamMap batch ({len(batch)} rows = {len(FOREST_PYROME)} forests × 5 percentiles)"
      f"\n  LCP column is BLANK — fill before running."
      f"\n  → {batch_fp}")

batch[["Forest", "Pyrome", "Scenario", "LCP", "WIND_SPEED", "WIND_DIRECTION",
       "FM_1hr", "FM_10hr", "FM_100hr", "FM_herb", "FM_woody"]]

In [ ]:
# ── Execute the batch — Windows VM only (TestFlamMap.exe) ─────────────────
# run_batch writes FlamMap.input (ShortTerm-Inputs) + FMcommand.txt per row and
# runs the exe, organising outputs under OUTPUT_ROOT/<lcp_stem>/<scenario>/.
# run_mtt()/run_fspro() guard non-Windows with RuntimeError; run FlamMap on the VM.
from fb_tools.models import run_batch

FM_EXE      = r"C:\FlamMap\TestFlamMap.exe"          # <-- Windows path to the exe
OUTPUT_ROOT = projdir / "data/mod/flammap_R4"

# results = run_batch(
#     FM_EXE, batch, OUTPUT_ROOT,
#     n_process=2,           # threads per FlamMap run
#     stack_out=True,        # stack per-output TIFFs into one multi-band file
#     cleanup=True,          # drop single-band TIFFs after stacking
#     skip_existing=True,    # resume: skip rows whose output already exists
# )
# results